# Continuum channels in streamed passes (find_continuum_ultra)

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/imaging/find_continuum_ultra.ipynb)

`find_continuum_ultra` finds the continuum channels of one spectral window of a dirty cube as the ALMA pipeline's `hif_findcont` does with findContinuum: a joint mask from the moment 0 and moment 8 images, the mean spectrum in the mask, a channel selection, and rounds that amend the mask (Hunter et al. 2023, *The ALMA Interferometric Pipeline Heuristics*). It reads the cube only in passes, map-reduces over the window's frequency chunks, and works on two-dimensional maps and spectra in the driver between them. The imager can make the first round while it writes the cube, in place of the first pass.

This notebook simulates a small ALMA data set with spectral lines at known channels in two spectral windows, images it, finds the continuum channels of each window with and without the imager's first round, and compares them with the simulated lines and with `find_continuum`, the port of the pipeline's findContinuum. It runs in less than a minute.

---
## Pseudo Code

```
distributed_applications/imaging/find_continuum_ultra(input_image_store, atmospheric_transmission,
                                                      atmospheric_sky_temperature, work_directory,
                                                      channel_range, first_round_store, ...)
    open the spectral window lazily; findContinuum's settings from its metadata
    first round: read first_round_store, or a pass over the residual and the primary beam
                 (primary-beam masks, per-channel rows, moments 0 and 8)
    joint mask from the moments                                      (driver)
    rounds of findContinuum, two passes each (time-division mode: the spectrum pass only)
        pass: mean spectrum in the joint mask   -> channel selection     (driver)
        pass: moments of the selected channels  -> decisions, the next joint mask (driver)
    write the joint mask; return the continuum channel ranges, flags, diagnostics
    and the records of the passes, the timing and the memory
```

## Notes

- One call per spectral window: `channel_range` is the window's `[start, stop)` in channels of the cube, at least 8 channels on a linear frequency axis.
- The atmospheric transmission and sky temperature of every channel are inputs; the pipeline takes them from CASA's atmosphere model.
- `image_cube_single_field(find_continuum_first_round=True)` writes a first-round store next to the cube: the primary-beam masks and per-channel rows of every channel and the moments of every spectral window. With it, `find_continuum_ultra` leaves out its first pass, the one that reads the primary beam with the residual.
- The passes over the residual are 3 on findContinuum's common path, 2 in time-division mode and 5 with an amended or extra joint mask, one fewer with the imager's first round.
- The selections, flags and decisions are those of `find_continuum`; with other frequency chunks or the imager's moments the moment-difference signal-to-noise ratio can differ in its last digits.

## Install AstroVIPER

Skip this cell when AstroVIPER is already installed.

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401 -- availability probe
except ImportError:
    os.system("pip install --upgrade astroviper[all]")
    import astroviper  # noqa: F401 -- verify the installation

print("Using astroviper version", version("astroviper"))

---
## API

In [ ]:
from astroviper.distributed_applications.imaging import (
    find_continuum,
    find_continuum_ultra,
    image_cube_single_field,
)

find_continuum_ultra?

## Example: two simulated ALMA spectral windows

### A Dask cluster

Four worker processes with one thread each. `find_continuum_ultra` takes the threads of the cluster and their memory for the frequency chunks of its passes and the memory budget of a task. The log level is WARNING to keep the output short; at INFO the applications log every pass and their timing.

In [ ]:
from toolviper.dask.client import local_client

from astroviper.utils.data_partitioning import get_thread_info

viper_client = local_client(
    cores=4, memory_limit="4GB", log_params={"log_level": "WARNING"}
)
get_thread_info()

### The simulated data set

ALMA antennas observe the field around its transit. A continuum point source sits at the phase centre, and four spectral lines (Gaussian in frequency, Gaussian sources on the sky) lie at known channels of two spectral windows with a gap between them. The table lists the lines; the simulator adds thermal noise.

In [ ]:
import numpy as np
import pandas as pd

ARCSEC = np.pi / (180.0 * 3600.0)
PHASE_CENTER = np.array([1.2, -0.4])  # radians
N_ANTENNA, N_CHANNELS = 20, 160
CHANNEL_WIDTH = 0.9765625e6  # Hz
CONTINUUM_FLUX_JY = 0.015
WINDOWS = [
    {"name": "spw_0", "first_frequency": 230.0e9},
    {"name": "spw_1", "first_frequency": 231.5e9},
]
LINES = pd.DataFrame(
    [
        (0, 0.08, 40, 8, 0.0, 0.0, 1.0, 0.8),
        (0, 0.15, 112, 20, 4.0, -3.0, 4.0, 3.0),
        (1, 0.06, 70, 12, 0.0, 0.0, 1.0, 0.8),
        (1, 0.05, 128, 4, -2.0, 2.5, 0.6, 0.6),
    ],
    columns=[
        "window",
        "peak_flux_jy",
        "channel",
        "fwhm_channels",
        "offset_east_arcsec",
        "offset_north_arcsec",
        "major_arcsec",
        "minor_arcsec",
    ],
)


def line_spectrum(line):
    """Flux (Jy) of a line at every channel of its window."""
    sigma = line.fwhm_channels / np.sqrt(8.0 * np.log(2.0))
    channels = np.arange(N_CHANNELS)
    return line.peak_flux_jy * np.exp(-0.5 * ((channels - line.channel) / sigma) ** 2)


LINES

In [ ]:
import os
import shutil

from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.simulation import simulate_processing_set
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout


def correlations(spectrum):
    """XX, XY, YX and YY flux of an unpolarized spectrum."""
    zeros = np.zeros_like(spectrum)
    return np.stack([spectrum, zeros, zeros, spectrum], axis=-1)


def simulate_window(index, ps_store):
    """The continuum source and the lines of one spectral window."""
    components = [
        {
            "kind": "point",
            "flux": correlations(np.full(N_CHANNELS, CONTINUUM_FLUX_JY)),
            "ra_dec": PHASE_CENTER,
        }
    ]
    for line in LINES[LINES.window == index].itertuples():
        offset = np.array(
            [
                line.offset_east_arcsec / np.cos(PHASE_CENTER[1]),
                line.offset_north_arcsec,
            ]
        )
        components.append(
            {
                "kind": "gaussian",
                "flux": correlations(line_spectrum(line)),
                "ra_dec": PHASE_CENTER + offset * ARCSEC,
                "major": line.major_arcsec * ARCSEC,
                "minor": line.minor_arcsec * ARCSEC,
                "pa": 0.5,
            }
        )
    return simulate_processing_set(
        ps_store=ps_store,
        antenna_xds=read_telescope_layout(
            "alma.cycle8.1", antenna_selection=list(range(N_ANTENNA))
        ),
        time_params={
            "time_start": "2026-03-01T22:12:00.000",
            "time_delta": 120.0,
            "n_samples": 15,
        },
        frequency_params={
            "freq_start": WINDOWS[index]["first_frequency"],
            "freq_delta": CHANNEL_WIDTH,
            "n_channels": N_CHANNELS,
            "channel_width": CHANNEL_WIDTH,
            "spectral_window_name": WINDOWS[index]["name"],
        },
        polarization=["XX", "YY"],
        sky_components=components,
        phase_center_ra_dec=PHASE_CENTER[None, :],
        beam_models=[airy_disk_model("alma")],
        beam_model_map=np.zeros(N_ANTENNA, int),
        noise_params={"random_seed": 11 + index},
        n_time_chunks=1,
        n_frequency_chunks=2,
        overwrite=True,
    )


ps_store = "find_continuum_ultra_demo.ps.zarr"
# Each window adds its MSv4 to a new processing set.
shutil.rmtree(ps_store, ignore_errors=True)
simulate_window(0, ps_store)
simulate_window(1, ps_store)
ps_xdt = open_processing_set(ps_store)
ps_xdt.xr_ps.summary()[
    ["name", "spw_name", "shape", "start_frequency", "end_frequency"]
]

### The dirty cube and the imager's first round

`max_iter=0` makes the dirty cube of both windows. With `find_continuum_first_round=True` the imager also makes findContinuum's first round of every channel as it writes the cube, into a store next to it, at the `primary_beam_limit` of `cycle_control_params`. It keeps the variables find_continuum reads: the residual, the primary beam and the beam fits of the point spread function.

In [ ]:
image_store = "find_continuum_ultra_demo.img.zarr"
frequencies = ps_xdt.xr_ps.get_freq_axis().values
imaging = image_cube_single_field(
    ps_store=ps_store,
    image_store=image_store,
    image_params={
        "image_size": [128, 128],
        "cell_size": np.array([-0.35, 0.35]) * ARCSEC,
        "phase_direction": PHASE_CENTER,
        "frequency_coords": frequencies,
        "polarization_coords": ["I", "Q"],
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    },
    imaging_weights_params={
        "weighting": "briggs",
        "robust": 0.5,
        "casa_weighting_implementation": True,
    },
    cycle_control_params={
        "max_iter": 0,
        "max_cycles": 0,
        "threshold": 0.0,
        "gain": 0.1,
        "psf_sidelobe_factor": 1.5,
        "max_iter_per_cycle": -1,
        "min_psf_fraction": 0.05,
        "max_psf_fraction": 0.8,
        "primary_beam_limit": 0.2,
    },
    image_data_variables_keep=[
        "sky_residual",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ],
    processing_set_data_group_name="base",
    n_mapping_parallelism={"frequency": 4},
    image_chunking={"frequency": 16},
    find_continuum_first_round=True,
    overwrite=True,
)
imaging["find_continuum_first_round"]

### Continuum channels of each window

The frequency axis of the cube is linear within each window, so `linear_channel_ranges` gives the windows' channels. The windows have no atmospheric line, and a flat atmosphere serves for both. Each call reads its window from the imager's first-round store.

In [ ]:
from astroviper.processing_functions.imaging.utils.spectral_windows import (
    linear_channel_ranges,
)

window_ranges = linear_channel_ranges(frequencies)
WORK = "find_continuum_ultra_demo_work"
shutil.rmtree(WORK, ignore_errors=True)


def atmosphere(window_range):
    """A flat atmosphere over the channels of a window."""
    n_channels = window_range[1] - window_range[0]
    return {
        "atmospheric_transmission": np.full(n_channels, 0.92),
        "atmospheric_sky_temperature": np.full(n_channels, 27.0),
    }


with_store = [
    find_continuum_ultra(
        input_image_store=image_store,
        work_directory=os.path.join(WORK, "imager_first_round"),
        channel_range=window_range,
        first_round_store=imaging["find_continuum_first_round"]["store"],
        monitor_resources_seconds=0.05,
        **atmosphere(window_range),
    )
    for window_range in window_ranges
]


def table(columns):
    """A table of the columns ``{name: {row: value}}``, every cell as given."""
    return pd.DataFrame(
        {name: pd.Series(rows, dtype=object) for name, rows in columns.items()}
    )


def results_column(result):
    """The results of a run as one table column."""
    return {
        "channels of the cube": result["channel_range"],
        "continuum channel ranges": result["continuum_channel_ranges"],
        "continuum frequency ranges (GHz)": [
            [round(frequency / 1e9, 6) for frequency in pair]
            for pair in result["continuum_frequency_ranges"]
        ],
        "aggregate bandwidth (MHz)": result["aggregate_bandwidth"] / 1e6,
        "all continuum": result["all_continuum"],
        "low bandwidth": result["low_bandwidth"],
        "low spread": result["low_spread"],
        "moment difference SNR": result["moment_difference_snr"],
        "joint mask": result["joint_mask_kind"],
        "passes": result["n_passes"],
        "passes reading each variable": result["cube_reads"],
    }


table(
    {
        window["name"]: results_column(result)
        for window, result in zip(WINDOWS, with_store, strict=True)
    }
)

### The spectrum and its selection

For the last round of each window: the spectrum its channel selection read, made from the mean of the residual in the joint mask, with the thresholds and the corrected median of the selection and the continuum ranges shaded.

In [ ]:
%matplotlib inline
from IPython.display import display

from astroviper.utils.report_plots import plot_continuum_selection

for window, result in zip(WINDOWS, with_store, strict=True):
    diagnostics = result["diagnostics"]
    spectrum = diagnostics["spectrum"]
    display(
        plot_continuum_selection(
            spectrum.frequency.values,
            spectrum.values,
            result["continuum_channel_ranges"],
            diagnostics["positive_threshold"],
            diagnostics["negative_threshold"],
            diagnostics["corrected_median"],
            title=window["name"],
        )
    )

### Against the simulated lines

A channel is line-free where every line of its window is below 1 % of its peak. The table counts the selected channels that are line-free and gives the brightest line in a selected channel as a fraction of that line's peak; the figure shows the continuum ranges under the line profiles.

In [ ]:
import matplotlib.pyplot as plt

LINE_FREE_LEVEL = 0.01  # of a line's peak
LINE_COLORS = ["#0072B2", "#E69F00"]


def channel_mask(ranges):
    """The channels of [start, stop) ranges as a mask over the window."""
    mask = np.zeros(N_CHANNELS, dtype=bool)
    for start, stop in ranges:
        mask[start:stop] = True
    return mask


def mask_ranges(mask):
    """The [start, stop) runs of a mask."""
    edges = np.diff(np.concatenate([[0], mask.astype(int), [0]]))
    return list(
        zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1), strict=True)
    )


truth = {}
fig, axes = plt.subplots(
    2, 2, figsize=(10, 4.5), sharex=True, height_ratios=[3, 1], layout="constrained"
)
for index, (window, result) in enumerate(zip(WINDOWS, with_store, strict=True)):
    window_lines = list(LINES[LINES.window == index].itertuples())
    levels = np.array(
        [line_spectrum(line) / line.peak_flux_jy for line in window_lines]
    )
    line_free = (levels < LINE_FREE_LEVEL).all(axis=0)
    selected = channel_mask(result["continuum_channel_ranges"])
    truth[window["name"]] = {
        "selected channels": int(selected.sum()),
        "of them line-free": int((selected & line_free).sum()),
        "line-free channels": int(line_free.sum()),
        "line-free channels selected (%)": round(
            100 * (selected & line_free).sum() / line_free.sum(), 1
        ),
        "brightest line in a selected channel (% of its peak)": round(
            100 * levels[:, selected].max(), 1
        ),
    }
    top, bottom = axes[:, index]
    for color, line in zip(LINE_COLORS, window_lines, strict=True):
        top.plot(
            1e3 * line_spectrum(line),
            color=color,
            linewidth=1.5,
            label=f"line at channel {line.channel}",
        )
    top.set_title(window["name"])
    top.set_ylabel("line flux (mJy)")
    top.legend(fontsize=8)
    top.grid(True, color="#d0d7de", linewidth=0.5)
    for row, (ranges, color) in enumerate(
        [
            (result["continuum_channel_ranges"], "#009E73"),
            (mask_ranges(line_free), "#8c959f"),
        ]
    ):
        bottom.broken_barh(
            [(start - 0.5, stop - start) for start, stop in ranges],
            (row + 0.15, 0.7),
            color=color,
        )
    bottom.set_yticks([0.5, 1.5], ["continuum ranges", "line-free"])
    bottom.set_xlabel("channel of the window")
plt.show()
table(truth)

### Without the imager's first round

Without a first-round store `find_continuum_ultra` makes the first round in a pass of its own, which reads the primary beam with the residual, and keeps that store in its run directory. For both runs on the first window, the first table counts the passes and the bytes they read; the second lists the passes: what each read, in how many frequency tasks, and the chunks it read. A pass leaves out the frequency chunks without a channel it needs.

In [ ]:
own_first_round = find_continuum_ultra(
    input_image_store=image_store,
    work_directory=os.path.join(WORK, "own_first_round"),
    channel_range=window_ranges[0],
    monitor_resources_seconds=0.05,
    **atmosphere(window_ranges[0]),
)
runs = {
    "imager's first round": with_store[0],
    "own first round": own_first_round,
}
print(
    "same continuum channel ranges:",
    own_first_round["continuum_channel_ranges"]
    == with_store[0]["continuum_channel_ranges"],
)
display(
    table(
        {
            name: {
                "passes": result["n_passes"],
                **{
                    f"passes reading {variable}": count
                    for variable, count in result["cube_reads"].items()
                },
                "residual bytes read (MB)": round(
                    result["passes"]["bytes_read_sky"].sum() / 1e6, 2
                ),
                "primary-beam bytes read (MB)": round(
                    result["passes"]["bytes_read_primary_beam"].sum() / 1e6, 2
                ),
            }
            for name, result in runs.items()
        }
    )
)
PASS_COLUMNS = [
    "pass_kind",
    "round_kind",
    "purpose",
    "data_variables_read",
    "n_tasks",
    "n_channels_read",
    "n_chunks_read",
    "bytes_read_sky",
    "bytes_read_primary_beam",
]
pd.concat(
    {
        name: result["passes"].set_index("pass_index")[PASS_COLUMNS]
        for name, result in runs.items()
    }
)

### Timing and memory

The driver's time per step, the memory plan of the frequency tasks with the largest resident memory of a worker process during a pass (from GraphVIPER's resource monitor), and the driver's peak. The plan of a task is a bound: a fixed baseline for the worker process plus the buffers of its pass. Last, the cost of the first round inside the imager against the pass that makes it.

In [ ]:
from astroviper.distributed_applications.imaging.find_continuum_ultra import (
    DISTRIBUTED_APPLICATION_TIMING_PHASES,
)

((_, _, steps),) = DISTRIBUTED_APPLICATION_TIMING_PHASES
display(
    pd.DataFrame(
        {
            name: {
                label: result["timing_distributed_application"][key]
                for label, key in steps
            }
            | {"total": result["timing_distributed_application"]["T_total"]}
            for name, result in runs.items()
        }
    ).round(3)
)
MIB = 2**20


def mib(value):
    return round(float(value) / MIB, 1)


display(
    table(
        {
            name: {
                "frequency tasks per pass": result["memory_plan"]["n_frequency_tasks"],
                "channels per task": result["memory_plan"]["channels_per_task"],
                "memory per thread (MiB)": mib(
                    result["memory_plan"]["memory_per_thread_gib"] * 2**30
                ),
                "task budget (MiB)": mib(result["memory_plan"]["budget_bytes"]),
                "largest task plan (MiB)": mib(
                    result["passes"]["task_memory_plan_bytes"].max()
                ),
                "largest worker RSS in a pass (MiB)": mib(
                    result["passes"]["task_rss_peak_bytes"].max()
                ),
                "driver RSS peak (MiB)": mib(
                    result["timing_distributed_application"]["driver_rss_peak_bytes"]
                ),
                "driver bytes read (MiB)": mib(
                    result["timing_distributed_application"]["driver_bytes_read"]
                ),
            }
            for name, result in runs.items()
        }
    )
)
imager_costs = with_store[0]["first_round"]["imager"]
first_pass = own_first_round["passes"].iloc[0]
pd.Series(
    {
        "imager: first round in its tasks (s, summed over tasks)": imager_costs[
            "T_find_continuum_first_round_sum"
        ],
        "imager: drain of the moments (s)": imager_costs["T_drain"],
        "find_continuum_ultra: first pass (s)": first_pass["T_compute_dask_graph"]
        + first_pass["T_merge_compute_dask_graph"],
    }
).round(3)

### Comparison with find_continuum

`find_continuum` runs the pipeline's findContinuum on the same window, on the frequency chunks of `find_continuum_ultra`, with a pass of its own for every reduction findContinuum asks for.

In [ ]:
port = find_continuum(
    input_image_store=image_store,
    work_directory=os.path.join(WORK, "find_continuum"),
    channel_range=window_ranges[0],
    n_mapping_parallelism={
        "frequency": own_first_round["memory_plan"]["n_frequency_tasks"]
    },
    **atmosphere(window_ranges[0]),
)


def selection_ranges(selection):
    """The [start, stop) ranges of a CASA channel selection "a~b;c~d"."""
    pairs = (pair.split("~") for pair in selection.split(";"))
    return [[int(first), int(last) + 1] for first, last in pairs]


comparison = {
    f"find_continuum_ultra, {name}": {
        "continuum channel ranges": result["continuum_channel_ranges"],
        "moment difference SNR": f"{result['moment_difference_snr']:.15g}",
        "aggregate bandwidth (MHz)": result["aggregate_bandwidth"] / 1e6,
        "all continuum": result["all_continuum"],
        "passes": result["n_passes"],
        "passes reading each variable": result["cube_reads"],
        "run time (s)": round(result["timing_distributed_application"]["T_total"], 2),
    }
    for name, result in runs.items()
}
comparison["find_continuum"] = {
    "continuum channel ranges": selection_ranges(port["channel_selection"]),
    "moment difference SNR": f"{port['moment_difference_snr']:.15g}",
    "aggregate bandwidth (MHz)": port["aggregate_bandwidth"] * 1e3,
    "all continuum": port["all_continuum"],
    "passes": port["n_cube_reductions"],
    "passes reading each variable": port["cube_reads"],
    "run time (s)": round(port["timing_distributed_application"]["T_total"], 2),
}
table(comparison)